# Lecture 4 Lab: Leakage-Safe Pipelines and Bioinformatics Decisions
**BINF 6210/8210 - Machine Learning for Bioinformatics**

Learning goals: detect leakage; compare transformations inside cross-validation; handle skew and outliers; recognize batch, repeated-measure, and high-dimensional pitfalls.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)


In [ ]:
n = 240
df = pd.DataFrame({
    "sample_id": [f"S{i:03d}" for i in range(n)],
    "age": rng.normal(52, 14, n).clip(18, 85),
    "bmi": rng.normal(27, 5, n).clip(16, 48),
    "batch": rng.choice(["B1", "B2", "B3"], n, p=[.45,.35,.20]),
    "sex": rng.choice(["Female", "Male"], n),
    "metabolite_a": rng.lognormal(1.2, .8, n),
    "metabolite_b": rng.lognormal(.5, .6, n),
})
logit = -3 + .045*df.age + .7*np.log1p(df.metabolite_a) + .35*(df.sex=="Male")
df["case"] = rng.binomial(1, 1/(1+np.exp(-logit)))
for col, frac in {"bmi":.08,"metabolite_a":.12,"batch":.04}.items():
    df.loc[rng.choice(n, int(n*frac), replace=False), col] = np.nan
df.loc[5, "bmi"] = 120  # deliberate data-quality issue
df.head()


## 1. Build one object that contains the full modeling workflow


In [ ]:
X=df.drop(columns=["case","sample_id"]); y=df.case
numeric=["age","bmi","metabolite_a","metabolite_b"]; categorical=["batch","sex"]
preprocess=ColumnTransformer([
 ("num",Pipeline([("impute",SimpleImputer(strategy="median")),("scale",RobustScaler())]),numeric),
 ("cat",Pipeline([("impute",SimpleImputer(strategy="most_frequent")),("encode",OneHotEncoder(handle_unknown="ignore"))]),categorical)])
model=Pipeline([("prep",preprocess),("clf",LogisticRegression(max_iter=1000))])
model


## 2. Cross-validation re-fits preprocessing within each training fold


In [ ]:
scores=cross_validate(model,X,y,cv=5,scoring=["roc_auc","accuracy"],return_train_score=True)
pd.DataFrame(scores).filter(regex="train|test").agg(["mean","std"]).T


## 3. Skewed abundance data often needs a scientific transformation

`log1p` is useful for nonnegative, right-skewed quantities and safely handles zero. Verify units, zeros, and negative values before using it.


In [ ]:
fig,ax=plt.subplots(1,2,figsize=(9,3))
df.metabolite_a.hist(ax=ax[0],bins=30); ax[0].set_title("Raw")
np.log1p(df.metabolite_a).hist(ax=ax[1],bins=30); ax[1].set_title("log1p")
plt.tight_layout()


In [ ]:
log_cols=["metabolite_a","metabolite_b"]
other_num=["age","bmi"]
preprocess_log=ColumnTransformer([
 ("other",Pipeline([("impute",SimpleImputer(strategy="median")),("scale",RobustScaler())]),other_num),
 ("abundance",Pipeline([("impute",SimpleImputer(strategy="median")),("log",FunctionTransformer(np.log1p,feature_names_out="one-to-one")),("scale",StandardScaler())]),log_cols),
 ("cat",Pipeline([("impute",SimpleImputer(strategy="most_frequent")),("encode",OneHotEncoder(handle_unknown="ignore"))]),categorical)])
model_log=Pipeline([("prep",preprocess_log),("clf",LogisticRegression(max_iter=1000))])
for name,m in {"robust-only":model,"log+scale":model_log}.items():
 print(name, cross_validate(m,X,y,cv=5,scoring="roc_auc")["test_score"].mean())


## 4. Group discussion: biology changes the split

If multiple samples come from the same participant, animal, plate, family, or site, ordinary random splitting can put related observations in both train and test sets. Name the grouping variable and choose a group-aware splitter.


In [ ]:
# Sketch only: groups must come from real metadata
# from sklearn.model_selection import GroupKFold
# cv = GroupKFold(n_splits=5)
# cross_validate(model, X, y, groups=participant_id, cv=cv)


## 5. Preprocessing decision record

For each step, document: input columns, fitted parameters, biological rationale, training-only boundary, and inverse interpretation (if any).


In [ ]:
decision_record=pd.DataFrame(columns=["step","columns","learned_from","rationale","risk_check"])
decision_record


## Exit ticket

Explain why a pipeline is more than convenient syntax. Name one batch-related variable that should be adjusted, stratified, grouped, or excluded - and justify your choice.
